In [ ]:
%%capture
!pip install transformers bitsandbytes accelerate peft trl datasets

In [ ]:
import torch

# GPU check
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


CUDA available: True
Device: Tesla T4
VRAM: 15.6 GB


In [ ]:
# HF login
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("collab-project-token")
if hf_token:
  login(hf_token)

In [ ]:
# Load model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model... (this may take 1-2 minutes)")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# add a dedicated pad token
tokenizer.add_special_tokens({'pad_token': '<pad>'})
model.resize_token_embeddings(len(tokenizer)) # model must know about the new token

tokenizer.padding_side = "right"

print("Model loaded successfully!")
print(f"Parameters: {model.num_parameters() / 1e9:.2f}B")

Loading model... (this may take 1-2 minutes)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded successfully!
Parameters: 3.21B


In [ ]:
# Load dataset
from datasets import load_dataset

ds = load_dataset("lavita/medical-qa-datasets", "medical_meadow_medqa")
print(ds)

README.md: 0.00B [00:00, ?B/s]

medical_meadow_medqa/train-00000-of-0000(…):   0%|          | 0.00/5.51M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 10178
    })
})


In [ ]:
# Inspect samples
for i in range(5):
  sample = ds['train'][i]
  print(f"--- Sample {i + 1} ---")
  print(f"INSTRUCTION: {sample['instruction']}")
  print(f"INPUT: {sample['input']}")
  print(f"OUTPUT: {sample['output'][:300]}")
  print()

--- Sample 1 ---
INSTRUCTION: Please answer with one of the option in the bracket
INPUT: Q:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?? 
{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': 'Ciprofloxacin', 'D': 'Doxycycline', 'E': 'Nitrofurantoin'},
OUTPUT: E: Nitrofurantoin

--- Sample 2 ---
INSTRUCTION: Please answer with one of the option in the bracket
INPUT: Q:A 3-month-old baby died suddenly at night while asleep. His mother noticed that he had died only af

In [ ]:
# Inference function
def generate_response(prompt):
  messages = [
      {"role": "system", "content": "You are a helpful medical assistant"},
      {"role": "user", "content": prompt}
  ]

  encoded_inputs = tokenizer.apply_chat_template(
      messages,                   # the conversation to format
      add_generation_prompt=True, # add the assistant header so the model knows to respond
      return_tensors="pt",        # package the output as a Pytorch tensor
      return_dict=True,           # returns a dict with input_ids AND attention_mask
      padding=True                # ensures attention mask is generated
  ).to(model.device)              # move that tensor to GPU memory where the model lives

  input_ids = encoded_inputs["input_ids"]
  attention_mask = encoded_inputs["attention_mask"]
  pad_token_id = tokenizer.pad_token_id

  outputs = model.generate(
      input_ids,                      # here is the prompt
      attention_mask=attention_mask,
      max_new_tokens=1000,             # generate at most 256 new tokens
      do_sample=True,                 # sample probabilistically, not greedily
      temperature=0.7,                # slightly sharpen the distribution - focused but not robotic
      top_p=0.9,                      # only sample from the top 90% probability tokens
      pad_token_id=pad_token_id       # tell generate which token is padding
  )

  response = outputs[0][input_ids.shape[-1]:] # cut off the input, keep only the generated answer
  return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
questions= [
    "What are the early symptoms of type 2 diabetes?",
    "what is the first-line treatment for hypertension in adults?",
    "What causes iron deficiency anemia?",
    "How is malaria diagnosed and treated?",
    "What are the warning signs of a heart attack?"
]

baseline_results = {}

for q in questions:
  print(f"Q: {q})")
  response = generate_response(q)
  baseline_results[q] = response
  print(f"A: {response}")
  print("=" * 60)

Q: What are the early symptoms of type 2 diabetes?)
A: assistant

As a medical assistant, I'd be happy to help you identify the early symptoms of type 2 diabetes.

Type 2 diabetes is often referred to as "silent diabetes" because it can be asymptomatic for a long time, especially in the early stages. However, some people may experience the following early symptoms:

1. **Increased thirst and hunger**: When your body produces more insulin, it can cause your body to hold onto more water, leading to increased thirst and hunger.
2. **Frequent urination**: As your body tries to flush out excess glucose, you may need to urinate more frequently, especially at night.
3. **Fatigue**: High blood sugar levels can cause fatigue, which can be mistaken for other conditions.
4. **Blurred vision**: High blood sugar levels can cause the lens in your eye to swell, leading to blurred vision.
5. **Slow healing of cuts and wounds**: High blood sugar levels can impede the healing process, leading to slower 